## 03_jackknife — 空間ジャックナイフ誤差評価

**入力**
- `output/data/coles_locations.csv`  — Coles 店舗座標 (N_C = 685)
- `output/data/random_masked.csv`    — 共有大陸マスク済みランダムカタログ (N_R = 10,390)
- `output/xi/xi_cc.csv`             — ξ_CC(r) Poisson 誤差（比較用）

**出力**
- `output/errors/jackknife.csv`     — r_km, xi_cc, sigma_poisson, sigma_jk, jk_over_poisson

**前提**
- `00_random_catalog.ipynb`, `01_xi_cc.ipynb` 実行済み
- `gradle :lib:jar` 完了

**手法**
$$\sigma_{\rm JK}^2(r) = \frac{K-1}{K} \sum_{k=0}^{K-1} \left[\hat{\xi}_k(r) - \bar{\xi}_{\rm JK}(r)\right]^2$$

$K = 8$ 経度スライス（113–154° 等分割, $\Delta{\rm lon} \approx 5.1°$/パッチ）。
データ点・ランダムカタログともにパッチ内点を除外して leave-one-out LS 推定量 $\hat{\xi}_k$ を計算。

**主要結果（参考値）**
$S/N_{\rm JK}^{\rm BAO} = 2.46\sigma$ @ $r = 666$ km（Cole et al. 2005 の 2dFGRS と同水準）

In [1]:
@file:DependsOn("../../lib/build/libs/retail-utils-1.0.jar")

In [2]:
%use dataframe
%use lets-plot

import retail.*
import kotlin.math.*

In [3]:
// --- データ読み込み ---
val dfC      = DataFrame.readCSV("./output/data/coles_locations.csv")
val dfRandom = DataFrame.readCSV("./output/data/random_masked.csv")
val dfXiCC   = DataFrame.readCSV("./output/xi/xi_cc.csv")

val dataPoints:   List<Point> = dfC.rows().map      { Point(it["lat"] as Double, it["lon"] as Double) }
val randomPoints: List<Point> = dfRandom.rows().map { Point(it["lat"] as Double, it["lon"] as Double) }

val nD = dataPoints.size;  val nR = randomPoints.size
println("N_D = $nD  |  N_R = $nR")

// ξ_CC Poisson 誤差（比較用）
data class PoissonBin(val rCenter: Double, val xi: Double, val sigmaP: Double)
val poissonBins: List<PoissonBin> = dfXiCC.rows().map {
    PoissonBin(it["r_km"] as Double, it["xi"] as Double, it["xi_err"] as Double)
}

N_D = 685  |  N_R = 10390


In [4]:
// --- ビン設計（xi_cc.csv と完全一致）---
val meanNN = dataPoints.map { p -> dataPoints.filter { it !== p }.minOf { haversine(p, it) } }.average()
val bins   = logBins(rMin = meanNN / 2.0)
val nBins  = bins.size - 1
println("nBins = $nBins  |  rMin = %.2f km".format(meanNN / 2.0))

nBins = 38  |  rMin = 7.63 km


In [5]:
// --- パッチ割り当て (K=8 経度スライス) ---
val K = 8
val lonEdges = DoubleArray(K + 1) { k -> 113.0 + k * (154.0 - 113.0) / K }
fun lonToPatch(lon: Double) = ((lon - 113.0) / (154.0 - 113.0) * K).toInt().coerceIn(0, K - 1)

val dataPatch = dataPoints.map   { lonToPatch(it.lon) }
val randPatch = randomPoints.map { lonToPatch(it.lon) }

println("=== 店舗分布 ===")
(0 until K).forEach { k ->
    println("  Patch $k [lon ${lonEdges[k].toInt()}–${lonEdges[k+1].toInt()}°]: ${dataPatch.count { it == k }} 店舗")
}

=== 店舗分布 ===
  Patch 0 [lon 113–118°]: 85 店舗
  Patch 1 [lon 118–123°]: 3 店舗
  Patch 2 [lon 123–128°]: 0 店舗
  Patch 3 [lon 128–133°]: 7 店舗
  Patch 4 [lon 133–138°]: 22 店舗
  Patch 5 [lon 138–143°]: 29 店舗
  Patch 6 [lon 143–148°]: 209 店舗
  Patch 7 [lon 148–154°]: 330 店舗


In [6]:
// --- Leave-one-out ループ ---
println("ジャックナイフ計算開始 (K=$K)...")

val xiJK = Array(K) { DoubleArray(nBins) { Double.NaN } }

for (k in 0 until K) {
    val dataK: List<Point> = dataPoints.filterIndexed   { i, _ -> dataPatch[i] != k }
    val randK: List<Point> = randomPoints.filterIndexed { i, _ -> randPatch[i] != k }
    val nDk = dataK.size;  val nRk = randK.size

    if (nDk < 10) { println("  Patch $k: スキップ (nD=$nDk)"); continue }

    val ddK = pairCounts(dataK, null,  bins)
    val drK = pairCounts(dataK, randK, bins)
    val rrK = pairCounts(randK, null,  bins)

    val normDDk = nDk.toLong() * (nDk - 1) / 2
    val normDRk = nDk.toLong() * nRk
    val normRRk = nRk.toLong() * (nRk - 1) / 2

    landySzalay(ddK, drK, rrK, normDDk, normDRk, normRRk, bins).forEachIndexed { b, bin ->
        xiJK[k][b] = bin.xi
    }
    println("  Patch $k 完了 (nD=$nDk, nR=$nRk)")
}
println("完了")

// --- ジャックナイフ σ ---
val sigmaJK = jackknifeSigma(xiJK)

println()
println("%-8s  %-9s  %-9s  %-9s  %-8s".format("r [km]", "ξ_CC", "σ_Poisson", "σ_JK", "JK/P"))
println("-".repeat(55))
poissonBins.forEachIndexed { b, pb ->
    val sJ = sigmaJK[b]
    val ratio = if (!pb.sigmaP.isNaN() && pb.sigmaP > 0 && !sJ.isNaN()) sJ / pb.sigmaP else Double.NaN
    println("%-8.1f  %-9.4f  %-9.5f  %-9.4f  %.1fx".format(pb.rCenter, pb.xi, pb.sigmaP, sJ, ratio))
}

ジャックナイフ計算開始 (K=8)...
  Patch 0 完了 (nD=600, nR=9603)
  Patch 1 完了 (nD=682, nR=9303)
  Patch 2 完了 (nD=685, nR=9060)
  Patch 3 完了 (nD=678, nR=8990)
  Patch 4 完了 (nD=663, nR=8847)
  Patch 5 完了 (nD=656, nR=8598)
  Patch 6 完了 (nD=476, nR=8664)
  Patch 7 完了 (nD=355, nR=9665)
完了

r [km]    ξ_CC       σ_Poisson  σ_JK       JK/P    
-------------------------------------------------------
8.2       232.3009   10.80746   186.6659   17.3x
9.5       251.8541   10.49920   231.4047   22.0x
11.0      217.6717   7.76037    191.8670   24.7x
12.7      187.0023   5.65564    170.3728   30.1x
14.8      181.6368   4.85009    166.9315   34.4x
17.1      164.9913   3.86444    156.1356   40.4x
19.8      142.4958   2.88146    130.6226   45.3x
22.9      120.7910   2.11022    109.7027   52.0x
26.5      100.0610   1.51804    93.4992    61.6x
30.7      76.6338    1.00334    71.4389    71.2x
35.5      55.8458    0.63158    53.7367    85.1x
41.1      42.4625    0.42117    38.8900    92.3x
47.6      33.4836    0.29029   

In [7]:
// --- プロット: σ_JK vs σ_Poisson エラーバー比較 ---
val rVec  = poissonBins.map { it.rCenter }
val xiVec = poissonBins.map { it.xi }
val rLong  = rVec + rVec
val loLong = poissonBins.mapIndexed { b, pb -> pb.xi - pb.sigmaP } +
             poissonBins.mapIndexed { b, pb -> pb.xi - sigmaJK[b] }
val hiLong = poissonBins.mapIndexed { b, pb -> pb.xi + pb.sigmaP } +
             poissonBins.mapIndexed { b, pb -> pb.xi + sigmaJK[b] }
val lblLong = List(rVec.size) { "Poisson σ" } + List(rVec.size) { "Jackknife σ" }

letsPlot(mapOf("r" to rLong, "xi" to (xiVec + xiVec), "lo" to loLong, "hi" to hiLong, "lbl" to lblLong)) +
    geomRibbon(alpha = 0.15) { x = "r"; ymin = "lo"; ymax = "hi"; fill = "lbl" } +
    geomLine(size = 1.1, color = "#4682B4") { x = "r"; y = "xi" } +
    geomHLine(yintercept = 0.0, linetype = "dashed", color = "#888888") +
    geomVLine(xintercept = 71.3,  linetype = "dotted", color = "#666666") +
    geomVLine(xintercept = 666.0, linetype = "dotted", color = "#CC4444") +
    scaleXLog10(name = "r [km]") + scaleYContinuous(name = "ξ_CC(r)") +
    scaleFillManual(values = mapOf("Poisson σ" to "#4682B4", "Jackknife σ" to "#E87040")) +
    ggtitle("ξ_CC: Poisson σ vs Jackknife σ  (K=8 longitude patches)",
            "赤点線: Retail BAO r=666 km  |  S/N_JK = 2.46σ") +
    ggsize(800, 420)

<path d="M0.0 159.81030411071518 L0.0 159.81030411071518 L16.239938563366536 148.17647179191712 L32.47987712673313 170.49582124508973 L48.71981569009961 190.30821119588018 L64.9597542534662 194.03870431048847 L81.19969281683274 204.69694921771324 L97.43963138019927 218.88998906694877 L113.67956994356581 232.47699047276348 L129.91950850693235 245.36654429736015 L146.15944707029888 259.83969366191593 L162.39938563366547 272.6310542643585 L178.639324197032 280.8485927123343 L194.87926276039855 286.3556001623754 L211.11920132376508 291.85377081520016 L227.35913988713162 296.4065712822565 L243.59907845049815 300.01487627664886 L259.8390170138647 302.0500572909285 L276.0789555772312 303.56220766815085 L292.31889414059776 303.5625490913666 L308.5588327039643 304.5673824127616 L324.79877126733084 305.08707833457026 L341.0387098306975 305.8112491530746 L357.2786483940639 306.03993149107913 L373.51858695743056 305.64531501207 L389.758525520797 306.33983566020385 L405.99846408416363 306.50607960684766 L422.23840264753017 306.4281357980061 L438.4783412108966 306.387869134012 L454.7182797742631 306.5273247683574 L470.9582183376299 306.3555117448437 L487.1981569009963 304.823063170729 L503.43809546436273 305.8777721908017 L519.6780340277294 306.6287822701163 L535.917972591096 306.77747806409155 L552.1579111544625 306.64818353255197 L568.397849717829 306.4623351030464 L584.6377882811955 306.876097403363 L600.8777268445622 307.09265320722795 L600.8777268445622 307.092892162448 L584.6377882811955 306.87652671530856 L568.397849717829 306.46315506026326 L552.1579111544625 306.648882390046 L535.917972591096 306.77810436589243 L519.6780340277294 306.6296442829937 L503.43809546436273 305.8796938033186 L487.1981569009963 304.82673787177487 L470.9582183376299 306.3571698039412 L454.7182797742631 306.5288870284158 L438.4783412108966 306.3899353587935 L422.23840264753017 306.43039283151495 L405.99846408416363 306.50844271571987 L389.758525520797 306.34305647308935 L373.51858695743056 305.6514671985345 L357.2786483940639 306.0453736222502 L341.0387098306975 305.8185707255102 L324.79877126733084 305.099371187584 L308.5588327039643 304.5847267539791 L292.31889414059776 303.5896936443982 L276.0789555772312 303.5933681486871 L259.8390170138647 302.10034991980365 L243.59907845049815 300.09484187019507 L227.35913988713162 296.544064985427 L211.11920132376508 292.07976123050906 L194.87926276039855 286.70656920269 L178.639324197032 281.35780154953085 L162.39938563366547 273.3946513672948 L146.15944707029888 261.05275193661635 L129.91950850693235 247.2018943139461 L113.67956994356581 235.02830061762114 L97.43963138019927 222.37375103435141 L81.19969281683274 209.3691587236803 L64.9597542534662 199.9025877989269 L48.71981569009961 197.14602464363952 L32.47987712673313 179.8783009171508 L16.239938563366536 160.87026209852203 L0.0 172.87678685132374 Z" fill="rgb(70,130,180)" stroke-width="1.0" fill-opacity="0.14901960784313725">
 
 
 
 <path d="M0.0 53.50166055465343 L0.0 53.50166055465343 L16.239938563366536 14.636363636363626 L32.47987712673313 59.201044282545354 L48.71981569009961 90.7346688333285 L64.9597542534662 96.0584903209299 L81.19969281683274 112.64712686851658 L97.43963138019927 141.6688783642863 L113.67956994356581 167.43600912567513 L129.91950850693235 189.76279209139796 L146.15944707029888 217.26053367086715 L162.39938563366547 240.52837571460753 L178.639324197032 257.5936962597184 L194.87926276039855 269.2795455011992 L211.11920132376508 280.16439071088655 L227.35913988713162 290.2967201330034 L243.59907845049815 297.02781226496273 L259.8390170138647 299.9016277042333 L276.0789555772312 301.58006066959325 L292.31889414059776 301.3548184837747 L308.5588327039643 303.42279969422947 L324.79877126733084 304.19045152017975 L341.0387098306975 305.17158022646987 L357.2786483940639 305.56385138314346 L373.51858695743056 305.08433813115454 L389.758525520797 306.0827460020595 L405.99846408416363 306.06912452191335 L422.23840264753017 306.08177973299337 L438.4

In [8]:
// --- CSV 出力 + BAO サマリー ---
java.io.File("./output/errors/jackknife.csv").bufferedWriter().use { w ->
    w.appendLine("r_km,xi_cc,sigma_poisson,sigma_jk,jk_over_poisson")
    poissonBins.forEachIndexed { b, pb ->
        val sJ = sigmaJK[b]
        val ratio = if (pb.sigmaP > 0 && !sJ.isNaN()) sJ / pb.sigmaP else Double.NaN
        w.appendLine("${pb.rCenter},${pb.xi},${pb.sigmaP},$sJ,$ratio")
    }
}
println("保存: output/errors/jackknife.csv")

// BAO ビン要約
val baoIdx = poissonBins.indices.minByOrNull { abs(poissonBins[it].rCenter - 666.0) }!!
val bao = poissonBins[baoIdx]
println()
println("=== Retail BAO (r ≈ ${bao.rCenter.toInt()} km) ===")
println("  ξ_CC       = %.4f".format(bao.xi))
println("  σ_Poisson  = %.5f  → S/N_P  = %.1fσ".format(bao.sigmaP,  bao.xi / bao.sigmaP))
println("  σ_JK       = %.4f  → S/N_JK = %.2fσ".format(sigmaJK[baoIdx], bao.xi / sigmaJK[baoIdx]))
println("  σ_JK/σ_P   = %.0fx".format(sigmaJK[baoIdx] / bao.sigmaP))

保存: output/errors/jackknife.csv

=== Retail BAO (r ≈ 666 km) ===
  ξ_CC       = 3.2215
  σ_Poisson  = 0.00304  → S/N_P  = 1059.9σ
  σ_JK       = 1.3052  → S/N_JK = 2.47σ
  σ_JK/σ_P   = 429x
